# TackleNet — Usage Notebook

TackleNet is a dataset of 737 de-identified MP4 clips, with face privacy protection applied using RetinaFace + SAM 3.1 where required, and all clips validated for playback integrity, annotated with
SATT tackling-technique scores and First Point of Contact (FPOC) frame labels.

This notebook is **training-free**. It demonstrates how to:

1. Access the full dataset and 30-clip sample dataset
2. Inspect annotations and class balance
3. Load de-identified MP4 clips
4. Extract the locked 32-frame FPOC-aligned contact window
5. Verify the frozen train/validation/test splits
6. Use the 30-clip sample for quick inspection

**Binary risk label rule** — derived from `satt3_score` only:
- Risky (`satt3_risk_binary = 1`): `satt3_score` ∈ {0, 1}
- Safe  (`satt3_risk_binary = 0`): `satt3_score` ∈ {2, 3}

`satt_sum` is an auxiliary descriptive field; it is **not** used to derive the benchmark label.

**Manuscript:** *TackleNet: A Benchmark Dataset for Tackle Safety Assessment in American Football Practice Videos*  
**License:** CC BY 4.0  
**Kaggle (full dataset):** https://www.kaggle.com/datasets/ahsanzaidi786/tacklenet  
**Kaggle (30-clip sample):** https://www.kaggle.com/datasets/ahsanzaidi786/tacklenet-sample


---
## 0. Environment Setup

Recommended local environment:

```bash
conda create -n tacklenet python=3.10 -y
conda activate tacklenet
pip install pandas numpy matplotlib opencv-python jupyter
jupyter notebook tacklenet_usage.ipynb
```

If you already have a Python environment with these packages installed, you can use it directly. The notebook does not train a model and does not require a GPU.


In [ ]:
# Required packages (pre-installed on Kaggle; for local use):
#   pip install pandas numpy matplotlib opencv-python jupyter

import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display


---
## 1. Data Access

### Option A — Running inside a Kaggle notebook (recommended)

Attach the datasets from the sidebar:
- **Full dataset:** https://www.kaggle.com/datasets/ahsanzaidi786/tacklenet
- **30-clip sample:** https://www.kaggle.com/datasets/ahsanzaidi786/tacklenet-sample

Then set `RUNNING_ON_KAGGLE = True` in the path cell below.

### Option B — Running locally

Download via the Kaggle API:

```bash
# Full dataset (~25 GB)
kaggle datasets download ahsanzaidi786/tacklenet --unzip -p ./tacklenet_data

# 30-clip sample (~1 GB)
kaggle datasets download ahsanzaidi786/tacklenet-sample --unzip -p ./tacklenet_sample
```

Place the unzipped contents so the folder layout matches:

```
your_working_folder/
  tacklenet_usage.ipynb
  tacklenet_data/
    annotations_satt_fpoc.csv
    splits_frozen.csv
    split_policy.md
    README.md
    videos/
      001.mp4  002.mp4  ...  745.mp4
  tacklenet_sample/                  # optional
    annotations_sample.csv
    README.md
    videos/
      021.mp4  044.mp4  ...  726.mp4
```

Then set `RUNNING_ON_KAGGLE = False` in the path cell below.

### What to expect when running this notebook

- Section 2 loads and validates the annotation CSVs and frozen split file.
- Section 3 displays annotation and class-balance plots inline.
- Sections 4 and 5 load an MP4 clip and visualize the 32-frame FPOC window.
- Section 6 prints frozen split statistics.
- Section 7 runs only if the 30-clip sample dataset is present.


In [ ]:
# Set to True when running inside a Kaggle notebook with datasets attached.
# Set to False when running locally after downloading the datasets.
RUNNING_ON_KAGGLE = True

if RUNNING_ON_KAGGLE:
    FULL_ROOT   = "/kaggle/input/tacklenet"
    SAMPLE_ROOT = "/kaggle/input/tacklenet-sample"
else:
    FULL_ROOT   = "./tacklenet_data"
    SAMPLE_ROOT = "./tacklenet_sample"

ANNOT_PATH   = f"{FULL_ROOT}/annotations_satt_fpoc.csv"
SPLITS_PATH  = f"{FULL_ROOT}/splits_frozen.csv"
VIDEOS_DIR   = f"{FULL_ROOT}/videos"

SAMPLE_ANNOT  = f"{SAMPLE_ROOT}/annotations_sample.csv"
SAMPLE_VIDEOS = f"{SAMPLE_ROOT}/videos"

print(f"Running on Kaggle : {RUNNING_ON_KAGGLE}")
print(f"Full dataset root : {FULL_ROOT}")
print(f"Annotation file   : {ANNOT_PATH}")
print(f"Splits file       : {SPLITS_PATH}")
print(f"Videos folder     : {VIDEOS_DIR}")
print(f"Sample root       : {SAMPLE_ROOT}")
print()
print("Annotation file exists :", os.path.exists(ANNOT_PATH))
print("Splits file exists     :", os.path.exists(SPLITS_PATH))
print("Videos folder exists   :", os.path.isdir(VIDEOS_DIR))
print("Sample file exists     :", os.path.exists(SAMPLE_ANNOT))


---
## 2. Load & Validate Annotations

In [ ]:
required_files = [ANNOT_PATH, SPLITS_PATH]
missing_files = [p for p in required_files if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError(
        "Missing required dataset files. Download and unzip the full dataset, "
        "then place it next to this notebook as './tacklenet_data/'. "
        f"Missing: {missing_files}"
    )

if not os.path.isdir(VIDEOS_DIR):
    print(f"WARNING: Video folder not found yet: {VIDEOS_DIR}")
    print("Sections 4 and 5 require the full videos/ folder.")

ann    = pd.read_csv(ANNOT_PATH)
splits = pd.read_csv(SPLITS_PATH)

print(f"Annotations : {ann.shape}")
print(f"Splits      : {splits.shape}")
ann.head()


In [ ]:
# ── Inline annotation integrity checks ─────────────────────────────────────

REQUIRED_COLS = [
    "clip_id", "clip_filename", "Session_ID", "fpoc_frame",
    "satt1", "satt2", "satt3_score", "satt4", "satt5", "satt6",
    "satt_sum", "satt3_risk_binary",
]
missing_cols = [c for c in REQUIRED_COLS if c not in ann.columns]
assert not missing_cols, f"Missing annotation columns: {missing_cols}"

assert len(ann) == 737,                "Expected 737 rows"
assert ann["clip_id"].is_unique,       "clip_id is not unique"
assert ann["clip_filename"].is_unique, "clip_filename is not unique"

# SATT component ranges
SATT_COLS = ["satt1", "satt2", "satt3_score", "satt4", "satt5", "satt6"]
for c in SATT_COLS:
    assert ann[c].isin([0, 1, 2, 3]).all(), f"Unexpected values in {c}"

# satt_sum consistency
assert (ann["satt_sum"] == ann[SATT_COLS].sum(axis=1)).all(), (
    "satt_sum does not match the sum of SATT1–SATT6"
)

# Binary label derivation
assert ann["satt3_risk_binary"].isin([0, 1]).all(), "Unexpected values in satt3_risk_binary"
derived = ann["satt3_score"].isin([0, 1]).astype(int)
assert (derived == ann["satt3_risk_binary"]).all(), (
    "satt3_risk_binary does not match satt3_score rule (0/1 → risky, 2/3 → safe)"
)

# fpoc_frame: must be non-negative integer
assert ann["fpoc_frame"].notna().all(), "fpoc_frame contains missing values"
assert (ann["fpoc_frame"] >= 0).all(), "fpoc_frame contains negative values"
assert (ann["fpoc_frame"] == ann["fpoc_frame"].astype(int)).all(), (
    "fpoc_frame contains non-integer values (possible CSV float encoding)"
)
ann["fpoc_frame"] = ann["fpoc_frame"].astype(int)

# Split counts
EXPECTED_SPLITS = {"train": 515, "val": 111, "test": 111}
actual_splits   = splits["split"].value_counts().to_dict()
for k, v in EXPECTED_SPLITS.items():
    assert actual_splits.get(k, 0) == v, (
        f"Split '{k}': expected {v}, found {actual_splits.get(k, 0)}"
    )

# Annotation ↔ split alignment (outer merge — no silent drops)
merged = ann.merge(
    splits[["clip_id", "split"]],
    on="clip_id", how="outer", indicator=True, validate="one_to_one",
)
bad_merge = merged[merged["_merge"] != "both"]
assert len(bad_merge) == 0, (
    f"Annotation/split mismatch for {len(bad_merge)} rows:\n{bad_merge}"
)
merged = merged.drop(columns="_merge")

print("All annotation integrity checks passed.")
print(f"  Clips          : {len(ann)}")
print(f"  Sessions       : {ann['Session_ID'].nunique()}")
print(f"  Risky (1)      : {ann['satt3_risk_binary'].sum()}")
print(f"  Safe  (0)      : {(ann['satt3_risk_binary'] == 0).sum()}")
print(f"  satt_sum range : {ann['satt_sum'].min()}–{ann['satt_sum'].max()}")
print(f"  FPOC range     : {ann['fpoc_frame'].min()}–{ann['fpoc_frame'].max()}")
print(f"  Split sizes    : { {k: actual_splits[k] for k in ['train','val','test']} }")


### Video File Integrity

The cell below verifies every CSV-listed video file is present in `videos/`. An optional slower check also verifies each video opens and that `fpoc_frame` is within the decoded frame range.


In [ ]:
# Verify that every CSV-listed video is present.
video_paths = [os.path.join(VIDEOS_DIR, fn) for fn in ann["clip_filename"]]
missing_videos = [p for p in video_paths if not os.path.exists(p)]

print(f"Expected videos : {len(video_paths)}")
print(f"Missing videos  : {len(missing_videos)}")

if missing_videos:
    print("First missing examples:")
    for p in missing_videos[:10]:
        print(" ", p)

assert len(missing_videos) == 0, "Some CSV-listed videos are missing from videos/"


In [ ]:
# Optional slower check: verify each video opens and FPOC is within frame range.
# Set RUN_VIDEO_DECODE_CHECK = True to run (may take several minutes on 737 clips).
RUN_VIDEO_DECODE_CHECK = False

if RUN_VIDEO_DECODE_CHECK:
    bad = []
    for _, row in ann.iterrows():
        path = os.path.join(VIDEOS_DIR, row["clip_filename"])
        cap  = cv2.VideoCapture(path)
        if not cap.isOpened():
            bad.append((row["clip_filename"], "cannot_open", None, row["fpoc_frame"]))
            continue
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if row["fpoc_frame"] < 0 or row["fpoc_frame"] >= n_frames:
            bad.append((row["clip_filename"], "fpoc_out_of_range",
                        n_frames, row["fpoc_frame"]))

    print(f"Videos checked : {len(ann)}")
    print(f"Problems       : {len(bad)}")

    if bad:
        display(pd.DataFrame(
            bad,
            columns=["clip_filename", "problem", "frame_count", "fpoc_frame"]
        ).head(20))

    assert len(bad) == 0, "Some videos failed the decode/FPOC range check."
else:
    print("Skipped full video decode check. "
          "Set RUN_VIDEO_DECODE_CHECK = True to run it.")


**Note on the split design:** The frozen benchmark split is **clip-level stratified** on
`satt3_risk_binary` (seed 42, 70/15/15). It is not session-grouped because reliable session
identifiers were not available at the time of split construction. `Session_ID` is now provided
in the annotation file for transparency and post-hoc analysis, but the official benchmark
split remains clip-level. This limitation is discussed in the dataset paper.


---
## 3. Annotation Visualizations

In [ ]:
print("Binary label rule:")
print("  Risky (satt3_risk_binary=1) : satt3_score ∈ {0, 1}")
print("  Safe  (satt3_risk_binary=0) : satt3_score ∈ {2, 3}")
print()
print("satt_sum is an auxiliary descriptive field, NOT the source of the binary label.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts = ann["satt3_risk_binary"].value_counts().reindex([0, 1], fill_value=0)
labels = ["Safe (0)", "Risky (1)"]
colors = ["#4C72B0", "#DD8452"]
axes[0].bar(labels, counts.values, color=colors, width=0.4)
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Clips")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=11)

# SATT3 score distribution
score_counts = ann["satt3_score"].value_counts().reindex([0, 1, 2, 3], fill_value=0)
bar_colors   = ["#DD8452", "#DD8452", "#4C72B0", "#4C72B0"]
axes[1].bar([str(s) for s in score_counts.index], score_counts.values,
            color=bar_colors, width=0.4)
axes[1].set_title("SATT3 Score Distribution (source of binary label)")
axes[1].set_xlabel("satt3_score")
axes[1].set_ylabel("Clips")
risky_p = mpatches.Patch(color="#DD8452", label="Risky (score 0–1)")
safe_p  = mpatches.Patch(color="#4C72B0", label="Safe (score 2–3)")
axes[1].legend(handles=[risky_p, safe_p])
for i, v in enumerate(score_counts.values):
    axes[1].text(i, v + 1, str(v), ha="center", fontsize=10)

plt.suptitle("TackleNet — Annotation Overview", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(ann["fpoc_frame"], bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.set_title("FPOC Frame Distribution")
ax.set_xlabel("fpoc_frame (0-indexed frame number within clip)")
ax.set_ylabel("Clips")
plt.tight_layout()
plt.show()

print(f"FPOC range  : {ann['fpoc_frame'].min()} – {ann['fpoc_frame'].max()}")
print(f"Median FPOC : {ann['fpoc_frame'].median():.0f}")


In [ ]:
# satt_sum is auxiliary — shown here for descriptive purposes only
fig, ax = plt.subplots(figsize=(9, 3))
for label, color, name in [(0, "#4C72B0", "Safe"), (1, "#DD8452", "Risky")]:
    subset = ann[ann["satt3_risk_binary"] == label]["satt_sum"]
    ax.hist(subset, bins=18, alpha=0.7, color=color, edgecolor="white",
            linewidth=0.3, label=name)
ax.set_title("satt_sum Distribution by Risk Label  [auxiliary — not the label source]")
ax.set_xlabel("satt_sum (sum of SATT1–SATT6, range 0–18)")
ax.set_ylabel("Clips")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Session-level clip counts
# The split is clip-level stratified, not session-grouped (see note above)
session_counts = ann["Session_ID"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(session_counts.index.astype(str), session_counts.values,
       color="#4C72B0", width=0.6)
ax.set_title("Clips per Session  [informational — split is clip-level, not session-grouped]")
ax.set_xlabel("Session_ID")
ax.set_ylabel("Clips")
plt.tight_layout()
plt.show()

print(f"Sessions    : {ann['Session_ID'].nunique()}")
print(f"Min / Max   : {session_counts.min()} / {session_counts.max()} clips per session")
print(f"Mean        : {session_counts.mean():.1f} clips per session")


---
## 4. Load a Video Clip

In [ ]:
def load_clip(clip_filename: str, videos_dir: str, max_frames: int = None) -> np.ndarray:
    """Load an MP4 clip as a numpy array of shape (T, H, W, 3) in RGB uint8."""
    path = os.path.join(videos_dir, clip_filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Video file does not exist: {path}")

    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise FileNotFoundError(f"OpenCV cannot open video: {path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if max_frames is not None and len(frames) >= max_frames:
            break
    cap.release()

    if len(frames) == 0:
        raise ValueError(f"Video opened but decoded zero frames: {path}")

    return np.stack(frames)   # (T, H, W, 3)


def show_frame_grid(clip: np.ndarray, idxs: list[int], title: str = "",
                    label_prefix: str = "f") -> None:
    """Display a subset of frames from a clip as a horizontal grid."""
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.5 * len(idxs), 2.8))
    for ax, idx in zip(axes, idxs):
        ax.imshow(clip[idx])
        ax.set_title(f"{label_prefix}{idx}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
example_row = ann.iloc[0]
clip = load_clip(example_row["clip_filename"], VIDEOS_DIR, max_frames=64)

print(f"Clip ID     : {example_row['clip_id']}")
print(f"Filename    : {example_row['clip_filename']}")
print(f"Session     : {example_row['Session_ID']}")
print(f"Label       : {'Risky' if example_row['satt3_risk_binary'] else 'Safe'} "
      f"(satt3_risk_binary={example_row['satt3_risk_binary']}, "
      f"satt3_score={example_row['satt3_score']})")
print(f"Shape       : {clip.shape}  (T, H, W, C)")

# 8 evenly spaced frames using original clip-frame indices
idxs = np.linspace(0, len(clip) - 1, 8, dtype=int).tolist()
show_frame_grid(
    clip, idxs,
    title=f"Clip {example_row['clip_id']} — {'Risky' if example_row['satt3_risk_binary'] else 'Safe'}",
    label_prefix="f",
)


---
## 5. FPOC Frame Extraction

The **locked asymmetric TackleNet FPOC window**:

- **23 frames before FPOC** + FPOC frame + **8 frames after** = **32 frames total**
- FPOC is always placed at window index 23.
- The window is intentionally pre-contact heavy to capture approach mechanics.
- Boundary handling uses **anchored index clamping**:
  - frames before the clip start repeat the first valid frame
  - frames beyond the clip end repeat the last valid frame
- The function never zero-pads and never shifts the FPOC anchor.


In [ ]:
def extract_fpoc_window(
    clip: np.ndarray,
    fpoc_frame: int,
    pre: int = 23,
    post: int = 8,
) -> np.ndarray:
    """
    Extract the locked asymmetric TackleNet FPOC window.

    The window always contains pre+1+post = 32 frames, with the FPOC frame
    placed exactly at window index `pre` (index 23 for the default pre=23).

    Boundary handling: indices outside the valid clip range are clamped to
    the nearest valid frame (repeat-first at the start, repeat-last at the end).
    This guarantees FPOC is always at w23 regardless of clip length.

    Parameters
    ----------
    clip       : (T, H, W, C) uint8 numpy array  [must be a numpy array]
    fpoc_frame : 0-indexed frame of First Point of Contact
    pre        : frames before FPOC (default 23, locked for TackleNet)
    post       : frames after  FPOC (default 8,  locked for TackleNet)

    Returns
    -------
    window : (pre+1+post, H, W, C) = (32, H, W, C) uint8 array
    """
    if clip is None or len(clip) == 0:
        raise ValueError("Empty clip: no decoded frames.")
    if pre < 0 or post < 0:
        raise ValueError("pre and post must be non-negative.")
    if fpoc_frame < 0 or fpoc_frame >= len(clip):
        raise ValueError(
            f"fpoc_frame={fpoc_frame} is outside valid range [0, {len(clip)-1}]."
        )

    T           = len(clip)
    raw_indices = np.arange(fpoc_frame - pre, fpoc_frame + post + 1)  # always 32 values
    clamped     = np.clip(raw_indices, 0, T - 1)
    window      = clip[clamped]   # fancy index into numpy array; clip must be ndarray

    expected = pre + 1 + post
    assert window.shape[0] == expected, (
        f"Window size mismatch: got {window.shape[0]}, expected {expected}"
    )
    assert np.array_equal(window[pre], clip[fpoc_frame]), (
        f"FPOC frame is not at window index {pre}."
    )
    return window


In [ ]:
fpoc   = int(example_row["fpoc_frame"])
window = extract_fpoc_window(clip, fpoc_frame=fpoc)

print(f"Full clip   : {len(clip)} frames")
print(f"FPOC frame  : {fpoc}  (original clip index)")
print(f"Window shape: {window.shape}  (should be 32 x H x W x 3)")
print(f"FPOC sits at window index 23 (pre=23 frames of approach context)")

# Explicitly include window index 23 so the FPOC frame is always visible
FPOC_WIN_IDX = 23
idxs = [0, 4, 8, 12, 16, 20, FPOC_WIN_IDX, 31]

fig, axes = plt.subplots(1, len(idxs), figsize=(2.5 * len(idxs), 2.8))
for ax, idx in zip(axes, idxs):
    ax.imshow(window[idx])
    label = f"w{idx}" + (" ← FPOC" if idx == FPOC_WIN_IDX else "")
    ax.set_title(label, fontsize=8)
    ax.axis("off")

plt.suptitle(
    f"FPOC Window — Clip {example_row['clip_id']}  "
    f"(23 pre-contact frames + FPOC[w23] + 8 post-contact frames)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

# Note: w-indices are window-local (0–31); original clip indices differ
print(f"\nOriginal clip frame for w0  : {fpoc - 23}  (clamped to 0 if negative)")
print(f"Original clip frame for w23 : {fpoc}  (FPOC)")
print(f"Original clip frame for w31 : {fpoc + 8}  (clamped to {len(clip)-1} if beyond end)")


---
## 6. Split Statistics

In [ ]:
data = merged.copy()   # outer-merged with indicator already validated above

summary = (
    data.groupby("split")["satt3_risk_binary"]
    .agg(clips="count", risky="sum", safe=lambda x: (x == 0).sum())
    .reindex(["train", "val", "test"])
)
summary["risky_%"] = (summary["risky"] / summary["clips"] * 100).round(2)
print(summary.to_string())
print(f"\nTotal: {summary['clips'].sum()} clips")


---
## 7. Sample Dataset

The optional **30-clip sample dataset** contains one randomly selected clip per practice
session (seed 42). It is intended for quick pipeline verification only.

It is **not** a valid benchmark split and should not be used for evaluation. If the sample dataset is not present in `./tacklenet_sample/`, the following cells skip gracefully.


In [ ]:
sample_available = os.path.exists(SAMPLE_ANNOT)

if sample_available:
    sample_ann = pd.read_csv(SAMPLE_ANNOT)
else:
    sample_ann = None
    print(f"Sample annotation file not found: {SAMPLE_ANNOT}")
    print("Skipping optional sample-dataset checks.")

if sample_available:
    # Verify exactly one clip per session and session count matches full dataset
    assert sample_ann["Session_ID"].nunique() == ann["Session_ID"].nunique(), (
        f"Session count mismatch: sample has {sample_ann['Session_ID'].nunique()}, "
        f"full dataset has {ann['Session_ID'].nunique()}"
    )
    assert (sample_ann.groupby("Session_ID").size() == 1).all(), (
        "Sample contains more than one clip for some session"
    )

    print(f"Full dataset sessions : {ann['Session_ID'].nunique()}")
    print(f"Sample clips          : {len(sample_ann)}  (one per session, seed 42)")
    print(f"Risky (1)             : {sample_ann['satt3_risk_binary'].sum()}")
    print(f"Safe  (0)             : {(sample_ann['satt3_risk_binary'] == 0).sum()}")
    print()
    print("WARNING: Use the full dataset + splits_frozen.csv for all benchmark evaluation.")
    display(sample_ann.head())


In [ ]:
if sample_ann is None:
    print("Skipping sample visualization because the sample dataset is not present.")
else:
    fig, ax = plt.subplots(figsize=(12, 3))
    colors_map = {0: "#4C72B0", 1: "#DD8452"}
    sample_sorted = sample_ann.sort_values("Session_ID")
    bar_colors = [colors_map[r] for r in sample_sorted["satt3_risk_binary"]]
    ax.bar(sample_sorted["Session_ID"].astype(str),
           [1] * len(sample_sorted), color=bar_colors, width=0.6)
    ax.set_title("Sample Dataset — Selected Clip per Session  (Blue=Safe, Orange=Risky)")
    ax.set_xlabel("Session_ID")
    ax.set_yticks([])
    risky_p = mpatches.Patch(color="#DD8452", label="Risky")
    safe_p  = mpatches.Patch(color="#4C72B0", label="Safe")
    ax.legend(handles=[safe_p, risky_p])
    plt.tight_layout()
    plt.show()


In [ ]:
if sample_ann is None:
    print("Skipping sample clip display because the sample dataset is not present.")
elif not os.path.isdir(SAMPLE_VIDEOS):
    print(f"Skipping sample clip display because the sample videos folder is missing: {SAMPLE_VIDEOS}")
else:
    sample_row  = sample_ann.iloc[0]
    sample_clip = load_clip(sample_row["clip_filename"], SAMPLE_VIDEOS)

    idxs = np.linspace(0, len(sample_clip) - 1, 8, dtype=int).tolist()
    show_frame_grid(
        sample_clip, idxs,
        title=(f"Sample Clip {sample_row['clip_id']} "
               f"(Session {sample_row['Session_ID']}) — "
               f"{'Risky' if sample_row['satt3_risk_binary'] else 'Safe'}"),
        label_prefix="f",
    )


---
## Notes & Citation

### Known limitations
- The official benchmark split is **clip-level stratified**, not session-grouped (see Section 6 note).
- Absent clip IDs 056, 057, 120, 164, 179, 338, 564, 662 are excluded by design.
- Single-rater annotations (Scott Dietrich, Albright College, Ed.D., LAT, ATC).
- IRB: Kansas State University IRB-13710, Exempt Category 4, Subsection ii.

### Per-video integrity validation
The inline checks above validate annotation columns, label derivation, frozen split counts, and annotation/split alignment. For full release packaging, also verify that `videos/` contains the expected 737 MP4 files and that every `fpoc_frame` is within the decoded frame range for its corresponding clip.


### Citation
If you use TackleNet in your research, please cite:

```bibtex
@dataset{zaidi2026tacklenet_dataset,
  author    = {Zaidi, Ahsan and Dietrich, Scott and Shamir, Lior},
  title     = {{TackleNet}: A Benchmark Dataset for Tackle Safety Assessment in American Football Practice Videos},
  year      = {2026},
  publisher = {Kaggle},
  url       = {https://www.kaggle.com/datasets/ahsanzaidi786/tacklenet},
  license   = {CC BY 4.0},
  note      = {IRB: Kansas State University IRB-13710. Paper citation to be added separately upon publication.}
}
```
